
# Digital Noise Source


## Transmitting 1228.8MHz Gaussian noise broadband that is sampled at 3.6864GHz. 
Nyquist boundary at 1.8432GHz 
Carrier frequency (Fc) = 900MHz  

Firmware specifics -->
This is "LFSR32x8" gaussian noise generation firmware --  1228.8MHz broadband, Fs=3.6864GHz; Fref=409.6MHz
ADC sample rate 1.2288GHz 
required axi-stream 76.8MHz 
adc_clk_out 76.8MHz

DAC sample rate 3.6864GHz 
required axi-stream 307.2MHz 
dac_clk_out 230.4MHz 

-- IP v23_0 (32x8)is used -- 128 bits (4I-4Q components) 
In Transmitter module, 
-- Adder tree for the addition of all LFSRs changed with same delays at every stage of pipeline
-- Added extra pipelining in AddSub blocks in transmitter IP - "as=32"
-- Added delay on PPS net to start PPS little late - "d=64"
In firmware,
-- PPS given to reset the signal ; pps given to comparator input AJ13

-- rfdc is taking ref_clk (for both adc and dac) of 409.6MHz which makes Fs= 3.6864GHz and giving clk_dac_out = 230.4MHz option and required axi stream clock at 307.2MHz (clocking wizard derivation) with WNS +0.706ns and TNS 0.000ns (failed routes 0); WHS 0.010ns and THS 0.000ns

-- reported total on-chip power 13.464W -- Reported junction temp 36.3°C


In [2]:
pwd

'/home/xilinx/jupyter_notebooks/lfsr32x8_1228.8MHz_3.6864gsps_edgeresetpps_G_128bits'

In [1]:
import time
import threading
import ipywidgets as ipw
from IPython.display import display
from rfsoc4x2 import oled
#importing rfsoc_radio package
from rfsoc_radio.overlay import RadioOverlay

oled1=oled.oled_display()
oled1.write("Hi, I am PEACC!")
time.sleep(2)

#initialising the overlay by downloading the bitstream and executing the drivers
ol = RadioOverlay(run_test=False, debug_test=False, bitfile_name="/home/xilinx/jupyter_notebooks/lfsr32x8_1228.8MHz_3.6864gsps_edgeresetpps_G_128bits/rfsoc_radio.bit")

dac_freq= ol.dac_block.MixerSettings["Freq"]
oled1.write(f"Initial DAC freq\n{dac_freq} MHz")

# on-board temps and voltages logging -- 
stop_flag={"stop":False}

t=threading.Thread(target=ol.save_temp_voltages, args=(stop_flag,), daemon=True)
t.start()

# adding start and stop button for temp and voltage logging 
stop_button=ipw.Button(description="stop temp/voltage logging", button_style="danger")
#start_button=ipw.Button(description="start temp/volage logging", button_style="success")

def on_stop_clicked(button):
    stop_flag["stop"]=True 
    print("Stopped temperature and voltage logging.")
    
#def on_start_clicked(button):
#    stop_flag["stop"]=False
#    print("Started temperature and voltage logging")
    
stop_button.on_click(on_stop_clicked)
display(stop_button)

#start_button.on_click(on_start_clicked)
#display(start_button)

ol.dashboard()

Button(button_style='danger', description='stop temp/voltage logging', style=ButtonStyle())

Label(value='Initial Reported Frequency: 900.0 MHz')

Accordion(children=(VBox(children=(HBox(children=(FloatText(value=900.0, description='DAC Frequency (MHz):', s…

In [4]:
from xrfclk import set_ref_clks

# setting LMK and LMX chips output frequency given that clock config files are available in board memeory 
set_ref_clks(lmk_freq=200.0, lmx_freq=409.6)

#### u32 XRFdc_SetDACVOP(XRFdc *InstancePtr, u32 Tile_Id, u32 Block_Id, u32 uACurrent)
#### u32 XRFdc_DynamicPLLConfig(XRFdc *InstancePtr, u32 Type, u32 Tile_Id, u8 Source, double RefClkFreq, double SamplingRate);
#### u32 XRFdc_SetInterpolationFactor(XRFdc *InstancePtr, u32 Tile_Id, u32 Block_Id, u32 InterpolationFactor);
#### u32 XRFdc_GetInterpolationFactor(XRFdc *InstancePtr, u32 Tile_Id, u32 Block_Id, u32 *InterpolationFactorPtr);
#### u32 XRFdc_GetOutputCurr(XRFdc *InstancePtr, u32 Tile_Id, u32 Block_Id, u32 *OutputCurrPtr);

In [16]:
""" Changing sampling rate changes dac_clock_out options beacuse they are derived from dac sampling circuitry and since IF and samples per axis clock cycle 
    do not change , bandwidth changes as per given formulas --
    1. Sampling_rate= BW X Interpolation Factor 
    2. BW = axis_clock_rate (i.e. data rate) X #I or Q samples per axis clock cycle , where axis clock is derived from dac_clock_out with the help of clock synthesizer
    
    One way to go about it -- write wrapper function to access and change parameters of clock synthesizer used in firmware which derives axis_clock from dac_clock_out from PYNQ
    Another way -- do not touch sampling rate for now from PYNQ until we write wrapper function for non-standard IPs. 
    Once success in accessing IPs from pynq, we can change sampling rate by changing axis_clock_rate and interpolation factor, so that bandwidth remains constant
"""
import cffi 

interp_ptr=cffi.FFI().new("unsigned int *")

## it won't take block_id as an argument yelling about more arguments passed than it expected, and this wrapper id designed for dac tile and not dac block
## I guess it needs both dac blocks enabled in dac tile for this wrapper to work 
#try:
#    ol.dac_tile.GetInterpolationFactor(interp_ptr) 
#    interp=interp_ptr[0]
#    print(f"DAC tile {ol.dac_tile}, block {ol.dac_block} interpolation factor = {interp}x")
#except Exception as e:
#    print("DAC 1 digital path 2 not enabled in XRFdc_GetInterpolationFactor")

#ol.dac_tile.DynamicPLLConfig(1, 409.6, 3686.4)
#ol.dac_block.SetInterpolationFactor(3)

['ClockSource', 'DumpRegs', 'DynamicPLLConfig', 'FIFOStatus', 'FabClkOutDiv', 'GetInterpolationFactor', 'PLLConfig', 'PLLLockStatus', 'Reset', 'SetupFIFO', 'ShutDown', 'StartUp', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_call_function', '_call_function_implicit', '_index', '_parent', '_type', 'blocks']
['BlockStatus', 'CoarseDelaySettings', 'DACCompMode', 'DataPathMode', 'DecoderMode', 'EnabledInterrupts', 'FabRdVldWords', 'FabWrVldWords', 'GetConnectedIData', 'GetConnectedQData', 'GetOutputCurr', 'IMRPassMode', 'InterpolationFactor', 'InvSincFIR', 'MixerSettings', 'NyquistZone', 'OutputCurr', 'PwrMode', 'QMCSettings', 'ResetInternalFIFOWidth', 'ResetNCOPhase', 'SetDACVOP', 'SetInterpola

In [3]:
"""
    Reading DAC output current -- using function in "xrfdc_function.c" --> u32 XRFdc_GetOutputCurr(XRFdc *InstancePtr, u32 Tile_Id, u32 Block_Id, u32 *OutputCurrPtr)
"""
import cffi 

# Preparing an 'unsigned int *' pointer for the API to fill
currout_ptr=cffi.FFI().new("unsigned int *")

ol.dac_block.GetOutputCurr(currout_ptr)

# Reading the value 
currout=currout_ptr[0]
print(f"Output current on DAC Tile 0 Block 0 is = {currout*(10**-3):.3f} mA")

Output current on DAC Tile 0 Block 0 is = 19.993 mA


In [9]:
"""
    Changing DAC output power -- using function in "xrfdc_function.c" --> u32 XRFdc_SetDACVOP(XRFdc *InstancePtr, u32 Tile_Id, u32 Block_Id, u32 uACurrent)
"""

ol.dac_block.SetDACVOP(20000)

In [11]:
"""
    Starting up and shutting down dac tile of interest or in-use: 
    
    DAC Tile 228 | DAC Tile 229 | DAC Tile 230 | DAC Tile 231 
    dac_tiles[0] | dac_tiles[1] | dac_tiles[2] | dac_tiles[3]
    
    For RFSoC4x2, available tiles are -- DAC Tile 228 | DAC Tile 230 
    DAC Tile selected in overlay.py is DAC_Tile 230 i.e.
    dac_tile = ol.rf.dac_tiles[2]
"""
ol.dac_tile.StartUp()
#ol.dac_tile.ShutDown()

In [2]:
"""
    Reading on-chip temperatures and voltages -- 
    Network listing directory at /sys/bus/iio/devices/" in <iio:device0> file 
    iio driver handles all the ADC conversion 
    Typical junction temperature on 4x2 during normal ops - 40-65 deg celsius  
"""
import os
import time 
from datetime import datetime

#path for processing system (low power domain) temperature sensor register
pslpd_temp_path="/sys/bus/iio/devices/iio:device0/in_temp0_ps_temp_raw"
pslpd_temp_offset_path="/sys/bus/iio/devices/iio:device0/in_temp0_ps_temp_offset"
pslpd_temp_scale_path="/sys/bus/iio/devices/iio:device0/in_temp0_ps_temp_scale"
#path for processing system (high power domain) temperature sensor register
pshpd_temp_path="/sys/bus/iio/devices/iio:device0/in_temp1_remote_temp_raw"
pshpd_temp_offset_path="/sys/bus/iio/devices/iio:device0/in_temp1_remote_temp_offset"
pshpd_temp_scale_path="/sys/bus/iio/devices/iio:device0/in_temp1_remote_temp_scale"
#path for PL fabric die temperature sensor register
pl_temp_path="/sys/bus/iio/devices/iio:device0/in_temp2_pl_temp_raw"
pl_temp_offset_path="/sys/bus/iio/devices/iio:device0/in_temp2_pl_temp_offset"
pl_temp_scale_path="/sys/bus/iio/devices/iio:device0/in_temp2_pl_temp_scale"
#path for PL core supply voltage sensor register
pl_vccint_path = "/sys/bus/iio/devices/iio:device0/in_voltage2_vccint_raw"
pl_vccint_scale_path="/sys/bus/iio/devices/iio:device0/in_voltage2_vccint_scale"
#path for PL aux voltage sensor register 
pl_vccaux_path = "/sys/bus/iio/devices/iio:device0/in_voltage25_vccplaux_raw"
pl_vccaux_scale_path="/sys/bus/iio/devices/iio:device0/in_voltage25_vccplaux_scale"

# Create new file with start timestamp
start_time=datetime.now().strftime("%Y%m%d_%H%M%S")
log_file=f"/home/xilinx/temp_vcc_log_{start_time}.csv"

print(f"Logging to: {log_file}")

with open(log_file, 'w') as f:
    f.write("timestamp,ps_lpd_temp_c,ps_hpd_temp_c,pl_temp_c,pl_core_voltage_v,pl_aux_voltage_v\n")
    
while True:
    try:
        # temperature read
        with open(pslpd_temp_path, 'r') as f:
            raw_temp1=int(f.read().strip())
        with open(pslpd_temp_offset_path, 'r') as f:
            offset1= int(f.read().strip())
        with open(pslpd_temp_scale_path, 'r') as f:
            scale1= float(f.read().strip())
            
        # converting raw temp values to celsius
        ps_lpd_temp_c= (raw_temp1+offset1) * scale1 / 1000
        #print(f"PS (low power domain) temperature: {ps_lpd_temp_c:.2f} deg Celsius")
        
        #-------------------------------------------#
        
        with open(pshpd_temp_path, 'r') as f:
            raw_temp2= int(f.read().strip())
        with open(pshpd_temp_offset_path, 'r') as f:
            offset2= int(f.read().strip())
        with open(pshpd_temp_scale_path, 'r') as f:
            scale2= float(f.read().strip())
            
        ps_hpd_temp_c= (raw_temp2+offset2) * scale2 / 1000
        #print(f"PS (high power domain) temperature: {ps_hpd_temp_c:.2f} deg Celsius")
        
        #-------------------------------------------#
        
        with open(pl_temp_path, 'r') as f:
            raw_temp3= int(f.read().strip())
        with open(pl_temp_offset_path, 'r') as f:
            offset3= int(f.read().strip())
        with open(pl_temp_scale_path, 'r') as f:
            scale3= float(f.read().strip())

        # converting raw temp values to celsius
        pl_temp_c= (raw_temp3+offset3) * scale3 / 1000
        #print(f"PL temperature: {pl_temp_c:.2f} deg Celsius")
        
        #-------------------------------------------#
        
        # voltage read
        with open(pl_vccint_path, 'r') as f:
            pl_vccint_raw= float(f.read().strip())
        with open(pl_vccint_scale_path, 'r') as f:
            pl_vccint_scale= float(f.read().strip())

        # converting raw voltage values to volts
        pl_vccint_v= pl_vccint_raw*pl_vccint_scale / 1000
        #print(f"PL core voltage: {pl_vccint_v:.3f}V")
        
        #-----------------------------------------#
        
        with open(pl_vccaux_path, 'r') as f:
            pl_vccaux_raw= float(f.read().strip())
        with open(pl_vccaux_scale_path, 'r') as f:
            pl_vccaux_scale= float(f.read().strip())

        # converting raw voltage values to volts
        pl_vccaux_v= pl_vccaux_raw*pl_vccaux_scale / 1000
        #print(f"PL aux voltage: {pl_vccaux_v:.3f}V")
        
        ts= datetime.now().isoformat(timespec="seconds")
        print(f"{ts}| PS (LPD) temp = {ps_lpd_temp_c:.3f} deg C | PS (HPD) temp = {ps_hpd_temp_c:.3f} deg C |")
        print(f"PL temp = {pl_temp_c:.3f} deg Celsius | PL core voltage = {pl_vccint_v:.4f}V | PL aux voltage = {pl_vccaux_v:.4f}V")
        
        # appending to new file
        with open(log_file, 'a') as f:
            f.write(f"{ts},{ps_lpd_temp_c:.3f}deg C,{ps_hpd_temp_c:.3f}deg C,{pl_temp_c:.3f}deg C,{pl_vccint_v:.4f}V,{pl_vccaux_v:.4f}V\n")
       
    except FileNotFoundError:
        print("SYSMON not accessible via sysfs")
    except Exception as e:
        print(f"Error reading sensors: {e}")
        
    time.sleep(10)


Logging to: /home/xilinx/temp_vcc_log_20260226_155107.csv
2026-02-26T15:51:07| PS (LPD) temp = 43.085 deg C | PS (HPD) temp = 43.567 deg C |
PL temp = 42.837 deg Celsius | PL core voltage = 0.8548V | PL aux voltage = 1.8082V
2026-02-26T15:51:17| PS (LPD) temp = 42.137 deg C | PS (HPD) temp = 43.365 deg C |
PL temp = 42.308 deg Celsius | PL core voltage = 0.8557V | PL aux voltage = 1.8057V
2026-02-26T15:51:27| PS (LPD) temp = 43.552 deg C | PS (HPD) temp = 43.583 deg C |
PL temp = 41.655 deg Celsius | PL core voltage = 0.8561V | PL aux voltage = 1.8058V
2026-02-26T15:51:37| PS (LPD) temp = 42.355 deg C | PS (HPD) temp = 43.505 deg C |
PL temp = 42.401 deg Celsius | PL core voltage = 0.8562V | PL aux voltage = 1.8061V
2026-02-26T15:51:47| PS (LPD) temp = 42.246 deg C | PS (HPD) temp = 42.386 deg C |
PL temp = 41.826 deg Celsius | PL core voltage = 0.8557V | PL aux voltage = 1.8047V
2026-02-26T15:51:57| PS (LPD) temp = 44.204 deg C | PS (HPD) temp = 43.862 deg C |
PL temp = 40.925 deg Cel

KeyboardInterrupt: 

In [7]:
from xrfclk import set_ref_clks

# setting LMK and LMX chips output frequency given that clock config files are available in board memeory 
set_ref_clks(lmk_freq=245.76, lmx_freq=409.6)

In [18]:
from xrfclk import *

# checking all clock register values on-board 
xrfclk._Config #Reg_values are in decimal

In [17]:
from xrfclk import *
print(xrfclk.lmk_devices)
#print(xrfclk.lmk_devices[0]['spi_device'])
print(xrfclk.lmx_devices)
#print(xrfclk.lmx_devices[0]['spi_device'])

[{'spi_device': PosixPath('/dev/spidev0.0'), 'compatible': 'lmk04828', 'num_bytes': 3}]
[{'spi_device': PosixPath('/dev/spidev0.2'), 'compatible': 'lmx2594'}, {'spi_device': PosixPath('/dev/spidev0.1'), 'compatible': 'lmx2594'}]


In [2]:
import spidev 
#Open spi device for communication:
spi = spidev.SpiDev(0,0) 

def lmkTransfer(spi, address, data=0x00, read=True):
    """
        This function performs LMK04828 SPI transfer, either read or write.
        What follows is low-level implimentation details:
        Either transfer sends 3 bytes.
        The only difference between read and write
        is that a read ignores the sent data and has a meaningful return.
        See LMK04828 datasheet at https://www.ti.com/lit/ds/symlink/lmk04828.pdf
        p25 has a diagram of the SPI timing, describing the 3 bytes,
            where the read/write bit goes R=1 for read, 0 for write,
            how the address gets split up, and
            1st byte:   R   0   0  A12 A11 A10 A9  A8
            2nd byte:  A7  A6  A5  A4  A3  A2  A1  A0
            3rd byte:  D7  D6  D5  D4  D3  D2  D1  D0
    """
    read_bit = 0x80 if read else 0x00  # high bit 1 for read, 0 for write
    address_high = (address>>8)&(0x1f)  # higher 5 bits of address
    address_low = address&(0xff)  # low 8 bits of address 
    to_send = [read_bit|address_high, address_low, data] # 3 bytes to send
    readback = spi.xfer(to_send) # a 3-item list, each one byte
    # the first two returned items should be 0. The last is the data in read mode
    return readback[2]

def lmkWrite(spi, address, data):
    lmkTransfer(spi, address, data, read=False)  # readback is always 0

def lmkRead(spi, address, data=0x00):
    """
        On the RFSoC 4x2 board, the STATUS_LD2 pin, attached to wire LMK_LD2
        is connected to both the "PLL2 locked" LED and the SPI readback"

        See the bottom of the CLOCK II page, p16 of the schematic:
        https://www.realdigital.org/downloads/3ae3a2552d7da46e9041196c654cd63d.pdf

        This pin is, by default, configured to output whether PLL2 is locked, 
        but the can be reconfigured to 18 different functions, 
        including "SPI readback".
        See PLL2_LD_MUX on p97 of the LMK04828 datasheet at:
        https://www.ti.com/lit/ds/symlink/lmk04828.pdf

        If this function always returns 0xff or 0x00,
            the PLL2_LD pin is not configured for "SPI readback".
        Instead, in this configuration, 0xff means PLL2 locked 
            and the PLL2 LED is on, whereas and 0x00 means it's not.
    """
    return lmkTransfer(spi, address, data, read=True)

def lmkSPIreadbackMode(spi):
    """
        Put the PLL2_LD pin into "SPI readback" and "Output (push-pull)" mode.
        A side effect is that the PLL2 LED on the board will be mostly off
        except for flashes that are too brief to see when you do SPI reads.
    """
    lmkWrite(spi, address=0x16E, data=0b00111011)

def lmkPLL2lockedLEDmode(spi):
    """
        Put the PLL2_LD pin (back) into the "PLL2 DLD" and "Output (push-pull)" mode
    """
    lmkWrite(spi, address=0x16E, data=0b00010011)

In [13]:
from xrfclk import *
from xrfclk import set_ref_clks
    
for i in range(50):
    # setting LMK and LMX chips output frequency given that clock config files are available in board memeory 
    set_ref_clks(lmk_freq=245.76, lmx_freq=409.6)
    
    lmkSPIreadbackMode(spi)
    
    address_lmk=[0x000,0x002,0x003,0x004,0x005,0x006,0x00C,0x00D,0x100,0x101,0x102,0x103,0x104,0x105,0x106,0x107,0x108,
             0x109,0x10A,0x10B,0x10C,0x10D,0x10E,0x10F,0x110,0x111,0x112,0x113,0x114,0x115,0x116,0x117,0x118,0x119,
             0x11A,0x11B,0x11C,0x11D,0x11E,0x11F,0x120,0x121,0x122,0x123,0x124,0x125,0x126,0x127,0x128,0x129,0x12A,
             0x12B,0x12C,0x12D,0x12E,0x12F,0x130,0x131,0x132,0x133,0x134,0x135,0x136,0x137,0x138,0x139,0x13A,0x13B,
             0x13C,0x13D,0x13E,0x13F,0x140,0x141,0x142,0x143,0x144,0x145,0x146,0x147,0x148,0x149,0x14A,0x14B,0x14C,
             0x14D,0x14E,0x14F,0x150,0x151,0x152,0x153,0x154,0x155,0x156,0x157,0x158,0x159,0x15A,0x15B,0x15C,0x15D,
             0x15E,0x15F,0x160,0x161,0x162,0x163,0x164,0x165,0x171,0x172,0x17C,0x17D,0x166,0x167,0x168,0x169,0x16A,
             0x16B,0x16C,0x16D,0x16E,0x173,0x182,0x183,0x184,0x185,0x188,0x189,0x18A,0x18B,0x1FFD,0x1FFE,0x1FFF]

    print('total no. of addresses being read is: {}'.format(len(address_lmk)))
    dat_lmk=[]
    #Fetching from LMK reg:
    for index,addr in enumerate(address_lmk):
        dat_lmk.append(hex(lmkRead(spi, address=addr)))
        print(f'R{addr} ({hex(addr)}) --> {dat_lmk[index]}')
        #np.savez('\Users\dspradio\Desktop\DNS\readback_LMK\dat_lmk{}.npz'.format(i),)
    print(dat_lmk)
    
lmkPLL2lockedLEDmode(spi)
    


total no. of addresses being read is: 135
R0 (0x0) --> 0x10
R2 (0x2) --> 0x0
R3 (0x3) --> 0x6
R4 (0x4) --> 0xd0
R5 (0x5) --> 0x5b
R6 (0x6) --> 0x20
R12 (0xc) --> 0x51
R13 (0xd) --> 0x4
R256 (0x100) --> 0x6a
R257 (0x101) --> 0x55
R258 (0x102) --> 0x55
R259 (0x103) --> 0x1
R260 (0x104) --> 0x22
R261 (0x105) --> 0x0
R262 (0x106) --> 0x73
R263 (0x107) --> 0x3
R264 (0x108) --> 0x6a
R265 (0x109) --> 0x55
R266 (0x10a) --> 0x55
R267 (0x10b) --> 0x0
R268 (0x10c) --> 0x22
R269 (0x10d) --> 0x0
R270 (0x10e) --> 0xf0
R271 (0x10f) --> 0x30
R272 (0x110) --> 0x6a
R273 (0x111) --> 0x55
R274 (0x112) --> 0x55
R275 (0x113) --> 0x1
R276 (0x114) --> 0x22
R277 (0x115) --> 0x0
R278 (0x116) --> 0x73
R279 (0x117) --> 0x3
R280 (0x118) --> 0x6a
R281 (0x119) --> 0x55
R282 (0x11a) --> 0x55
R283 (0x11b) --> 0x1
R284 (0x11c) --> 0x22
R285 (0x11d) --> 0x0
R286 (0x11e) --> 0x72
R287 (0x11f) --> 0x3
R288 (0x120) --> 0x74
R289 (0x121) --> 0x55
R290 (0x122) --> 0x55
R291 (0x123) --> 0x1
R292 (0x124) --> 0x22
R293 (0x125) 

In [73]:
lmkSPIreadbackMode(spi)

In [74]:
address_lmk=[0x000,0x002,0x003,0x004,0x005,0x006,0x00C,0x00D,0x100,0x101,0x102,0x103,0x104,0x105,0x106,0x107,0x108,
             0x109,0x10A,0x10B,0x10C,0x10D,0x10E,0x10F,0x110,0x111,0x112,0x113,0x114,0x115,0x116,0x117,0x118,0x119,
             0x11A,0x11B,0x11C,0x11D,0x11E,0x11F,0x120,0x121,0x122,0x123,0x124,0x125,0x126,0x127,0x128,0x129,0x12A,
             0x12B,0x12C,0x12D,0x12E,0x12F,0x130,0x131,0x132,0x133,0x134,0x135,0x136,0x137,0x138,0x139,0x13A,0x13B,
             0x13C,0x13D,0x13E,0x13F,0x140,0x141,0x142,0x143,0x144,0x145,0x146,0x147,0x148,0x149,0x14A,0x14B,0x14C,
             0x14D,0x14E,0x14F,0x150,0x151,0x152,0x153,0x154,0x155,0x156,0x157,0x158,0x159,0x15A,0x15B,0x15C,0x15D,
             0x15E,0x15F,0x160,0x161,0x162,0x163,0x164,0x165,0x171,0x172,0x17C,0x17D,0x166,0x167,0x168,0x169,0x16A,
             0x16B,0x16C,0x16D,0x16E,0x173,0x182,0x183,0x184,0x185,0x188,0x189,0x18A,0x18B,0x1FFD,0x1FFE,0x1FFF]

print('total no. of addresses being read is: {}'.format(len(address_lmk)))
dat_lmk=[]
#Fetching from LMK reg:
for index,addr in enumerate(address_lmk):
    dat_lmk.append(hex(lmkRead(spi, address=addr)))
    print(f'R{addr} ({hex(addr)}) --> {dat_lmk[index]}')
print(dat_lmk)


total no. of addresses being read is: 135
R0 (0x0) --> 0x10
R2 (0x2) --> 0x0
R3 (0x3) --> 0x6
R4 (0x4) --> 0xd0
R5 (0x5) --> 0x5b
R6 (0x6) --> 0x20
R12 (0xc) --> 0x51
R13 (0xd) --> 0x4
R256 (0x100) --> 0x6a
R257 (0x101) --> 0x55
R258 (0x102) --> 0x55
R259 (0x103) --> 0x1
R260 (0x104) --> 0x22
R261 (0x105) --> 0x0
R262 (0x106) --> 0x73
R263 (0x107) --> 0x3
R264 (0x108) --> 0x6a
R265 (0x109) --> 0x55
R266 (0x10a) --> 0x55
R267 (0x10b) --> 0x0
R268 (0x10c) --> 0x22
R269 (0x10d) --> 0x0
R270 (0x10e) --> 0xf0
R271 (0x10f) --> 0x30
R272 (0x110) --> 0x6a
R273 (0x111) --> 0x55
R274 (0x112) --> 0x55
R275 (0x113) --> 0x1
R276 (0x114) --> 0x22
R277 (0x115) --> 0x0
R278 (0x116) --> 0x73
R279 (0x117) --> 0x3
R280 (0x118) --> 0x6a
R281 (0x119) --> 0x55
R282 (0x11a) --> 0x55
R283 (0x11b) --> 0x1
R284 (0x11c) --> 0x22
R285 (0x11d) --> 0x0
R286 (0x11e) --> 0x72
R287 (0x11f) --> 0x3
R288 (0x120) --> 0x74
R289 (0x121) --> 0x55
R290 (0x122) --> 0x55
R291 (0x123) --> 0x1
R292 (0x124) --> 0x22
R293 (0x125) 

In [75]:
##Writing values to LMK -- 
dct=xrfclk._Config['lmk04828'][245.76]
#print(dct)
hex_dct=[]
for i in range(len(dct)):
    hex_dct.append(hex(dct[i]))
print('total no. of addresses being written is: {}'.format(len(hex_dct[1:])))
print(hex_dct[1:])

total no. of addresses being written is: 135
['0x10', '0x200', '0x306', '0x4d0', '0x55b', '0x600', '0xc51', '0xd04', '0x1006a', '0x10155', '0x10255', '0x10301', '0x10422', '0x10500', '0x10673', '0x10703', '0x1086a', '0x10955', '0x10a55', '0x10b00', '0x10c22', '0x10d00', '0x10ef0', '0x10f30', '0x1106a', '0x11155', '0x11255', '0x11301', '0x11422', '0x11500', '0x11673', '0x11703', '0x1186a', '0x11955', '0x11a55', '0x11b01', '0x11c22', '0x11d00', '0x11e72', '0x11f03', '0x12074', '0x12155', '0x12255', '0x12301', '0x12422', '0x12500', '0x12670', '0x12733', '0x1286a', '0x12955', '0x12a55', '0x12b00', '0x12c22', '0x12d00', '0x12ef0', '0x12f30', '0x1306a', '0x13155', '0x13255', '0x13301', '0x13422', '0x13500', '0x13673', '0x13703', '0x13800', '0x13903', '0x13a01', '0x13b40', '0x13c00', '0x13d01', '0x13e03', '0x13f02', '0x14009', '0x14100', '0x14200', '0x14331', '0x144ff', '0x1457f', '0x1461b', '0x1470a', '0x14802', '0x14942', '0x14a06', '0x14b26', '0x14c00', '0x14d00', '0x14ec0', '0x14f7f', '0x

In [76]:
lmkPLL2lockedLEDmode(spi)

In [7]:
import spidev 
"""
    SPI devices:
    For PLL1 (ADC PLL) --> Bus=0; Device=1
    For PLL2 (DAC PLL) --> Bus=0; Device=2
"""
#Open spi device for communication:
"syntax -- spi=spidev.SpiDev(Bus,Device)"
spi = spidev.SpiDev(0,2) 

def lmxTransfer(spi, address, data=0x0000, read=True):
    """
        This function performs LMX SPI 2594 transfer, either read or write.
        What follows is low-level implimentation details:
        Either transfer sends 3 bytes.
        The only difference between read and write
        is that a read ignores the sent data and has a meaningful return.
        See LMX2594 datasheet at https://www.ti.com/lit/ds/symlink/lmx2594.pdf
        Describing the 3 bytes,
            where the read/write bit goes R=1 for read, 0 for write,
            how the data gets split up, and
            1st byte:  R/W  A6   A5   A4   A3   A2   A1  A0
            2nd byte:  D15  D14  D13  D12  D11  D10  D9  D8
            3rd byte:  D7   D6   D5   D4   D3   D2   D1  D0
    """
    read_bit = 0x80 if read else 0x00  # high bit 1 for read, 0 for write
    data_high = (data>>8)&(0xff)  # higher 8 bits of data
    data_low = data&(0xff) # lower 8 bits of data 
    to_send = [read_bit|address, data_high, data_low] # 3 bytes to send
    readback = spi.xfer(to_send) # a 3-item list, each one byte
    # the last two returned items are the data in read mode
    return readback[1],readback[2]

def lmxWrite(spi, address, data):
    lmxTransfer(spi, address, data, read=False)  # readback is always 0
    
def lmxRead(spi, address, data=0x0000):
    """
        On the RFSoC 4x2 board, the MUX_OUT pin from LMX2594, attached to wire RF_PLL2/1_MUX_OUT 
        is connected to both the "DAC/ADC locked" LED and "the SPI readback"

        See the bottom of the CLOCK II page, p16/17 of the schematic:
        https://www.realdigital.org/downloads/3ae3a2552d7da46e9041196c654cd63d.pdf

        This pin is, by default, configured to output whether DAC/ADC is locked, 
        but the can be reconfigured to SPI redback function, 
        0: SPI readback
        1: Lock Detect
        See MUXOUT_LD_SEL on p46 of the LMX2594 datasheet at:
        https://www.ti.com/lit/ds/symlink/lmx2594.pdf
    """
    return lmxTransfer(spi, address, data, read=True)

def lmxSPIreadbackMode(spi):
    """
        Put the MUXOUT_LD_SELECT (R0[2]) pin into "SPI readback" mode
        A side effect is that the DAC LED on the board will be mostly off
        except for flashes that are too brief to see when you do SPI reads.
        D15-D0: '0x2418'
    """
    lmxWrite(spi, address=0x00, data=0b0010010000011000)
    
def lmxDAClockedLEDmode(spi):
    """
        Put the MUXOUT_LD_SELECT (R0[2]) pin (back) into the "LockDetect" mode
        D15-D0: '0x241C'
    """
    lmxWrite(spi, address=0x00, data=0b0010010000011100)


In [13]:
from xrfclk import *
from xrfclk import set_ref_clks
from time import sleep 
    
for i in range(10):
    # setting LMK and LMX chips output frequency given that clock config files are available in board memeory 
    set_ref_clks(lmk_freq=245.76, lmx_freq=409.6)
    
    lmxSPIreadbackMode(spi)
    
    address_lmx=[0x70,0x6f,0x6e,0x14]

    print('total no. of addresses being read is: {}'.format(len(address_lmx)))
    for index,addr in enumerate(address_lmx):
        D1,D0=lmxRead(spi,address=addr)
        print(f'R{addr} ({hex(addr)}) --> {hex(D1)},{hex(D0)}')
    
    sleep(30)
lmxDAClockedLEDmode(spi)
    


total no. of addresses being read is: 4
R112 (0x70) --> 0x1,0x64
R111 (0x6f) --> 0x0,0x96
R110 (0x6e) --> 0x4,0x68
R20 (0x14) --> 0xf8,0x48
total no. of addresses being read is: 4
R112 (0x70) --> 0x1,0x64
R111 (0x6f) --> 0x0,0x96
R110 (0x6e) --> 0x4,0x68
R20 (0x14) --> 0xf8,0x48
total no. of addresses being read is: 4
R112 (0x70) --> 0x1,0x64
R111 (0x6f) --> 0x0,0x96
R110 (0x6e) --> 0x4,0x68
R20 (0x14) --> 0xf8,0x48
total no. of addresses being read is: 4
R112 (0x70) --> 0x1,0x64
R111 (0x6f) --> 0x0,0x96
R110 (0x6e) --> 0x4,0x68
R20 (0x14) --> 0xf8,0x48
total no. of addresses being read is: 4
R112 (0x70) --> 0x1,0x64
R111 (0x6f) --> 0x0,0x96
R110 (0x6e) --> 0x4,0x68
R20 (0x14) --> 0xf8,0x48
total no. of addresses being read is: 4
R112 (0x70) --> 0x1,0x64
R111 (0x6f) --> 0x0,0x96
R110 (0x6e) --> 0x4,0x68
R20 (0x14) --> 0xf8,0x48
total no. of addresses being read is: 4
R112 (0x70) --> 0x1,0x64
R111 (0x6f) --> 0x0,0x96
R110 (0x6e) --> 0x4,0x68
R20 (0x14) --> 0xf8,0x48
total no. of address

In [8]:
lmxSPIreadbackMode(spi)

In [9]:
#Add register addresses here::
address_lmx=[0x70,0x6f,0x6e,0x6d,0x6c,0x6b,0x6a,0x69,0x68,0x67,0x66,0x65,0x64,0x63,0x62,0x61,0x60,0x5f,0x5e,0x5d,0x5c,
            0x5b,0x5a,0x59,0x58,0x57,0x56,0x55,0x54,0x53,0x52,0x51,0x50,0x4f,0x4e,0x4d,0x4c,0x4b,0x4a,0x49,0x48,0x47,
            0x46,0x45,0x44,0x43,0x42,0x41,0x40,0x3f,0x3e,0x3d,0x3c,0x3b,0x3a,0x39,0x38,0x37,0x36,0x35,0x34,0x33,0x32,
            0x31,0x30,0x2f,0x2e,0x2d,0x2c,0x2b,0x2a,0x29,0x28,0x27,0x26,0x25,0x24,0x23,0x22,0x21,0x20,0x1f,0x1e,0x1d,
            0x1c,0x1b,0x1a,0x19,0x18,0x17,0x16,0x15,0x14,0x13,0x12,0x11,0x10,0x0f,0x0e,0x0d,0x0c,0x0b,0x0a,0x09,0x08,
            0x07,0x06,0x05,0x04,0x03,0x02,0x01,0x00]
#Fetching from LMX reg:
print('total no. of addresses being read is: {}'.format(len(address_lmx)))
for index,addr in enumerate(address_lmx):
    D1,D0=lmxRead(spi,address=addr)
    print(f'R{addr} ({hex(addr)}) --> {hex(D1)},{hex(D0)}')

total no. of addresses being read is: 113
R112 (0x70) --> 0x1,0x63
R111 (0x6f) --> 0x0,0x96
R110 (0x6e) --> 0x4,0x68
R109 (0x6d) --> 0x9d,0x7d
R108 (0x6c) --> 0x0,0xf2
R107 (0x6b) --> 0x88,0x1
R106 (0x6a) --> 0x0,0x0
R105 (0x69) --> 0x0,0x21
R104 (0x68) --> 0x0,0x0
R103 (0x67) --> 0x0,0x0
R102 (0x66) --> 0x3f,0x80
R101 (0x65) --> 0x0,0x11
R100 (0x64) --> 0x0,0x0
R99 (0x63) --> 0x0,0x0
R98 (0x62) --> 0x2,0x0
R97 (0x61) --> 0x8,0x88
R96 (0x60) --> 0x0,0x0
R95 (0x5f) --> 0x0,0x0
R94 (0x5e) --> 0x0,0x0
R93 (0x5d) --> 0x0,0x0
R92 (0x5c) --> 0x0,0x0
R91 (0x5b) --> 0x0,0x0
R90 (0x5a) --> 0x0,0x0
R89 (0x59) --> 0x0,0x0
R88 (0x58) --> 0x0,0x0
R87 (0x57) --> 0x0,0x0
R86 (0x56) --> 0x0,0x0
R85 (0x55) --> 0xd3,0x0
R84 (0x54) --> 0x0,0x1
R83 (0x53) --> 0x0,0x0
R82 (0x52) --> 0x1e,0x0
R81 (0x51) --> 0x0,0x0
R80 (0x50) --> 0x66,0x66
R79 (0x4f) --> 0x0,0x26
R78 (0x4e) --> 0x1,0x6f
R77 (0x4d) --> 0x0,0x0
R76 (0x4c) --> 0x0,0xc
R75 (0x4b) --> 0x9,0x80
R74 (0x4a) --> 0x0,0x0
R73 (0x49) --> 0x0,0x3f
R72 (

In [80]:
##Writing values to LMX -- 
dct1=xrfclk._Config['lmx2594'][409.6]
#print(dct1)
hex_dct1=[]
for i in range(len(dct1)):
    hex_dct1.append(hex(dct1[i]))
print('total no. of addresses being written is: {}'.format(len(hex_dct1)))
print(hex_dct1)

total no. of addresses being written is: 113
['0x700000', '0x6f0000', '0x6e0000', '0x6d0000', '0x6c0000', '0x6b0000', '0x6a0000', '0x690021', '0x680000', '0x670000', '0x663f80', '0x650011', '0x640000', '0x630000', '0x620200', '0x610888', '0x600000', '0x5f0000', '0x5e0000', '0x5d0000', '0x5c0000', '0x5b0000', '0x5a0000', '0x590000', '0x580000', '0x570000', '0x560000', '0x55d300', '0x540001', '0x530000', '0x521e00', '0x510000', '0x506666', '0x4f0026', '0x4e00e5', '0x4d0000', '0x4c000c', '0x4b0980', '0x4a0000', '0x49003f', '0x480001', '0x470081', '0x46c350', '0x450000', '0x4403e8', '0x430000', '0x4201f4', '0x410000', '0x401388', '0x3f0000', '0x3e0322', '0x3d00a8', '0x3c0000', '0x3b0001', '0x3a8001', '0x390020', '0x380000', '0x370000', '0x360000', '0x350000', '0x340820', '0x330080', '0x320000', '0x314180', '0x300300', '0x2f0300', '0x2e07fc', '0x2dc0df', '0x2c1f20', '0x2b0000', '0x2a0000', '0x290000', '0x280000', '0x270001', '0x260000', '0x250104', '0x240190', '0x230004', '0x220000', '0x211

In [81]:
hex(0b0010010000011000)

'0x2418'

In [82]:
hex(0b0010010000011100)

'0x241c'

In [83]:
lmxDAClockedLEDmode(spi)